# Name That LLM: offline submission (batch inference)

This notebook produces our competition submission. When you submit it, Kaggle **re-runs it
on ~4,000 hidden passages with the internet turned off**. While you edit it, it only sees 3
example test rows. The code must therefore work offline and must not assume a row count.

- **Code:** the project's package, attached as the dataset `ryqn00/name-that-llm-lib` and
  published automatically by CI on each version tag
  ([repo](https://github.com/Ryqn0/name-that-llm)).
- **Model:** `ENSEMBLE = "ensemble_p8"`, a log blend (a weighted geometric mean) of five
  models' probabilities, every weight chosen by nested cross-validation
  (`configs/ensemble_p8.yaml` holds the numbers):
  1. the TF-IDF text model `lr`, trained on our passages **plus the pool's allowlisted
     passages**;
  2. a logistic regression on **Qwen 3 4B's hidden states** (layers 6 to 9, averaged over
     each passage's tokens);
  3. **ModernBERT-base, fine-tuned** to name the model: the average of its 5 fold models;
  4. **Qwen3-8B with LoRA** on 4-bit weights, with 5,000 pool passages per fold: the average
     of its 5 fold adapters;
  5. **DeBERTa-v3-large**, trained once on the whole clean pool and never on one of our
     passages: the average of its 2 seeds.
- **Older recipes:** `ENSEMBLE = "ensemble_llm"` gives back public LB 0.505,
  `"ensemble_pool"` 0.454 and `"ensemble"` 0.410; `USE_ENCODER = False`, `lm_blend` (0.389).
- **The pool:** our private pinned copy of the clean pool v10
  (`ryqn00/name-that-llm-pool-v10`, from `crtproduction/name-that-llm-open-pool`, CC BY-SA
  4.0), rebuilt here offline with the project's safeguards: copies of our passages and
  conflicting labels removed, and only the sources that helped `lr` (3 left in v10). `lr`
  trains on all 28,048 of their passages, the count the gate measured.
- **On the GPU:** Qwen 3 4B is attached as a Kaggle Model
  (`qwen-lm/qwen-3/transformers/4b-base`, Apache 2.0) and runs in float16. The encoder's 5
  fold models (about 1.5 GB) are the encoder-final notebook's output, attached here, and
  DeBERTa's 2 seeds are notebook 13's, attached the same way. torch
  and transformers come preinstalled in Kaggle's GPU image, so our package doesn't depend
  on them.

In [ ]:
# Internet is OFF, so install from the attached dataset. Search for the wheel instead of
# hard-coding the mount path: Kaggle's input layout has changed before (the competition data
# now sits under /kaggle/input/competitions/...).
import glob
import os

found = glob.glob("/kaggle/input/**/name_that_llm-*.whl", recursive=True)
wheel_dirs = {os.path.dirname(p) for p in found}
assert len(wheel_dirs) == 1, f"expected exactly one wheel folder, found: {wheel_dirs}"
WHEELS = wheel_dirs.pop()
print("installing from", WHEELS, "->", sorted(os.listdir(WHEELS)))

In [ ]:
# --no-index: never reach for PyPI (there's no internet). --find-links: take wheels from
# the attached folder. Packages Kaggle already has (pandas, scikit-learn, ...) stay as they are.
%pip install -q --no-index --find-links {WHEELS} name-that-llm peft bitsandbytes

# P7: the LLM member needs peft (LoRA) and bitsandbytes (4-bit weights), shipped as wheels in
# the lib dataset. Stop at once, naming the missing one, rather than fail hours later.
import importlib

for package in ("peft", "bitsandbytes"):
    try:
        importlib.import_module(package)
    except ImportError as err:
        raise SystemExit(f"the lib dataset lacks {package}: {err}") from err

In [ ]:
import time

import numpy as np
import pandas as pd

import name_that_llm
from name_that_llm.config import BlendConfig, EncoderConfig, EnsembleConfig, load_config
from name_that_llm.data import load_test, load_train
from name_that_llm.submit import (
    predict_submission,
    predict_submission_blend,
    predict_submission_ensemble,
)

CHAMPION = "lr"  # the text-only champion: public LB 0.320
USE_LM = True  # P3b: blend lr with Qwen 3 hidden states (public LB 0.389). False = text-only lr
USE_ENCODER = True  # P5 A/B test: add the fine-tuned encoder. False = the P3b blend
BLEND = "lm_blend"  # the BlendConfig chosen by the sweep notebook (05-lm-sweep)
# P8: + DeBERTa. "ensemble_llm": LB 0.505; "ensemble_pool": 0.454; "ensemble": 0.410
ENSEMBLE = "ensemble_p8"
assert USE_LM or not USE_ENCODER, "the ensemble blends the LM head too: set USE_LM = True"

ens_cfg = load_config(ENSEMBLE, model_cls=EnsembleConfig) if USE_ENCODER else None
enc_cfg = load_config(ens_cfg.encoder, model_cls=EncoderConfig) if USE_ENCODER else None
# Each config names the one it was tuned with: the ensemble its blend, the blend its text model.
blend_cfg = (
    load_config(ens_cfg.blend if USE_ENCODER else BLEND, model_cls=BlendConfig) if USE_LM else None
)
text_cfg = load_config(blend_cfg.text_config if USE_LM else CHAMPION)
print("package", name_that_llm.__version__, "| text model:", text_cfg.name)
print("blend:", blend_cfg)
print("ensemble:", ens_cfg)

train, test = load_train(), load_test()  # validated: ids stay strings, labels are checked
print("train", train.shape, "| test", test.shape)

# I-029: the text model also trains on the pool, rebuilt here offline with the same
# safeguards as on the PC (pool.build_pool). Counts only.
pool = None
if USE_ENCODER and ens_cfg.text_pool:
    from pathlib import Path

    from name_that_llm.pool import build_pool, load_pool_sources

    # lr's allowlist (pool.yaml), read from our PINNED copy of the clean pool v10, not the
    # public dataset: its 2026-11-01 release shrank one of lr's sources and brought back
    # another (prvsiyan), so its latest version isn't what the gate measured.
    pinned = "ryqn00/name-that-llm-pool-v10"
    sources = [s.model_copy(update={"slug": pinned}) for s in load_pool_sources("pool")]
    pool = build_pool(sources, Path("/kaggle/input"), train)
    if len(pool) != 28_048:  # the gate's lr (P7 Task 7); any other pool is unmeasured
        raise SystemExit(f"STOP: lr's pool has {len(pool):,} passages, not the measured 28,048")
    print(f"pool: {len(pool):,} passages; lr trains on {min(ens_cfg.text_pool, len(pool)):,}")

## LM features on the GPU

Each passage becomes one vector: the average, over its tokens, of the model's hidden states
in the blend's layers. We extract them for train **and** test, here, in the same run.

**Why recompute the training features** instead of attaching the ones the
[features notebook](https://www.kaggle.com/code/ryqn00/name-that-llm-lm-features) saved? It
costs about 6 GPU-minutes, but it's one artifact less to keep in sync: train and test
features come from the same code, model and run, so they can't drift apart.

In [ ]:
extract_min = 0.0  # training-feature extraction time, added to the runtime check below
if USE_LM:
    import gc

    import torch
    import transformers

    from name_that_llm.lm import find_model_dir, load_lm, pooled_hidden_states, select_layers

    print("torch", torch.__version__, "| transformers", transformers.__version__)
    # Fail in the first seconds, not after hours: the blend needs the GPU (accelerator: T4).
    assert torch.cuda.is_available(), "USE_LM = True needs a GPU"
    LM_DIR = find_model_dir(blend_cfg.lm_variation)  # searched, never hard-coded

    # Keep only the layers the blend uses (hidden_states index: 0 = embeddings, k = block k).
    # The forward pass computes every layer anyway; this just saves memory.
    LAYERS = range(blend_cfg.layer, blend_cfg.layer + blend_cfg.layer_window)

    def lm_features(texts):
        # Loaded per call, and freed on return: Qwen 3 4B holds about 8 GB of the T4 in
        # float16, and the LLM needs that memory later. Version 6 kept it loaded, and the
        # LLM ran out of memory on the runtime check's 400 passages; the hidden test's
        # 4,000 would have too.
        model, tokenizer = load_lm(LM_DIR)
        feats = pooled_hidden_states(
            texts,
            model,
            tokenizer,
            layers=LAYERS,
            batch_size=blend_cfg.batch_size,
            max_length=blend_cfg.max_length,
        )
        del model
        gc.collect()
        torch.cuda.empty_cache()  # hand the memory back before the next model loads
        return select_layers(feats, 0, blend_cfg.layer_window)  # the mean of those layers

    start = time.time()
    lm_train = lm_features(train["text"].tolist())
    extract_min = (time.time() - start) / 60
    # STRICT on train: these are known passages, finite in the features notebook. A failure
    # here is a real problem, so stop before submitting anything.
    assert np.isfinite(lm_train).all(), "non-finite train features (see 04-lm-features)"

    lm_test = lm_features(test["text"].tolist())
    # LENIENT on test: float16 can overflow on an unusual passage, and one bad passage should
    # cost one prediction, not crash the 9-hour run. The TRAIN MEAN is the neutral input:
    # once scaled it's all zeros, so the LM head gives near-uniform probabilities and the
    # text model decides that passage.
    bad = ~np.isfinite(lm_test).all(axis=1)
    if bad.any():
        print(f"warning: {bad.sum()} test passages had non-finite LM features")
        lm_test[bad] = lm_train.mean(axis=0)
    print(f"LM features: train {lm_train.shape} in {extract_min:.1f} min, test {lm_test.shape}")

## The fine-tuned encoder on the GPU

The [encoder-final notebook](https://www.kaggle.com/code/ryqn00/name-that-llm-encoder-final)
fine-tuned ModernBERT-base once per cross-validation fold and saved the 5 models. Nothing is
trained here: the encoder's prediction is the **average of those 5 models**, the same ones
whose out-of-fold predictions chose the ensemble's weights.

In [ ]:
if USE_ENCODER:
    from name_that_llm.encoder import find_fold_dirs, predict_proba_folds

    # Searched, never hard-coded, and exactly 5 or an error: averaging 4 models would
    # silently change the submission.
    FOLD_DIRS = find_fold_dirs(name="encoder", n_folds=5)
    print("fold models:", [str(d) for d in FOLD_DIRS])

    def encoder_proba(texts):
        # The training max_length, so every passage is cut exactly as it was in CV.
        p = predict_proba_folds(
            FOLD_DIRS, texts, batch_size=2 * enc_cfg.batch_size, max_length=enc_cfg.max_length
        )
        # LENIENT, as with the LM features: one bad passage costs one prediction, not the
        # run. A uniform row adds the same amount to all 18 labels, so it can't change which
        # label wins: the other two models decide that passage.
        bad = ~np.isfinite(p).all(axis=1)
        if bad.any():
            print(f"warning: {bad.sum()} passages had non-finite encoder probabilities")
            p[bad] = 1 / p.shape[1]
        return p

    start = time.time()
    p_encoder = encoder_proba(test["text"].tolist())
    print(f"encoder: test {p_encoder.shape} in {(time.time() - start) / 60:.1f} min")

## The LLM on the GPU (P7)

Qwen3-8B with LoRA, fine-tuned on our 18 labels with 5,000 pool passages per fold
(notebook 12). Its 5 fold adapters, saved in a private dataset, are averaged on one 4-bit
base model: the same models whose out-of-fold predictions chose the ensemble's weights.

In [ ]:
p_llm = None
if USE_ENCODER and ens_cfg.llm is not None:
    from name_that_llm.config import LLMConfig
    from name_that_llm.encoder import find_fold_dirs
    from name_that_llm.llm import predict_proba_adapters
    from name_that_llm.lm import find_model_dir

    llm_cfg = load_config(ens_cfg.llm, model_cls=LLMConfig).model_copy(
        update={"base_model": str(find_model_dir("8b-base"))}
    )
    # Exactly 5, or an error: averaging 4 adapters would silently change the submission.
    ADAPTERS = find_fold_dirs(name="llm", n_folds=5, marker="adapter_config.json")
    print("adapters:", [str(d) for d in ADAPTERS])

    def llm_proba(texts):
        p = predict_proba_adapters(llm_cfg, ADAPTERS, texts, batch_size=2 * llm_cfg.batch_size)
        bad = ~np.isfinite(p).all(axis=1)  # lenient, as for the encoder
        if bad.any():
            print(f"warning: {bad.sum()} passages had non-finite LLM probabilities")
            p[bad] = 1 / p.shape[1]
        return p

    start = time.time()
    p_llm = llm_proba(test["text"].tolist())
    print(f"llm: test {p_llm.shape} in {(time.time() - start) / 60:.1f} min")

## DeBERTa-v3-large on the GPU (P8)

Trained once on the whole clean pool, and never on one of our passages (notebook 13, private
until the deadline). Nothing is trained here: the member's prediction is the **average of its
2 seeds' final models**, the same ones whose predictions on our passages chose the ensemble's
weights.


In [ ]:
p_extra = None
if USE_ENCODER and ens_cfg.extra_members:
    import json
    from pathlib import Path

    from name_that_llm.encoder import predict_proba_folds
    from name_that_llm.pool_member import find_member_dirs

    SEEDS = (42, 43)  # notebook 13's two runs
    extra_cfgs = {m: load_config(m, model_cls=EncoderConfig) for m in ens_cfg.extra_members}
    # Searched, never hard-coded, and every seed's final epoch or an error: one seed of two
    # would silently change the submission.
    EXTRA_DIRS = {m: find_member_dirs(SEEDS, c.epochs) for m, c in extra_cfgs.items()}
    print("pool members:", {m: [str(d) for d in dirs] for m, dirs in EXTRA_DIRS.items()})
    # The run the gate measured, and no other. Attached outputs come from one version of
    # notebook 13, and its version 2 still held a source we leave out (7,620 passages).
    counts_path = next(Path("/kaggle/input").rglob("pool_counts.json"))
    trained_on = sum(json.loads(counts_path.read_text(encoding="utf-8")).values())
    if trained_on != 190_887:
        raise SystemExit(f"STOP: this member's pool held {trained_on:,} passages, not 190,887")

    def extra_proba(texts):
        out = {}
        for member, dirs in EXTRA_DIRS.items():
            cfg = extra_cfgs[member]
            p = predict_proba_folds(
                dirs, texts, batch_size=2 * cfg.batch_size, max_length=cfg.max_length
            )
            bad = ~np.isfinite(p).all(axis=1)  # lenient, as for the encoder
            if bad.any():
                print(f"warning: {bad.sum()} passages had non-finite {member} probabilities")
                p[bad] = 1 / p.shape[1]
            out[member] = p
        return out

    start = time.time()
    p_extra = extra_proba(test["text"].tolist())
    print(f"pool members: test in {(time.time() - start) / 60:.1f} min")

In [ ]:
start = time.time()
# Fits on ALL of train (CV was only for measuring), predicts test in order, and VALIDATES
# the result (every id exactly once, no nulls, only the 18 labels) before we write anything.
if USE_ENCODER:
    sub = predict_submission_ensemble(
        text_cfg,
        blend_cfg,
        ens_cfg,
        train,
        test,
        lm_train,
        lm_test,
        p_encoder,
        pool=pool,
        p_llm=p_llm,
        p_extra=p_extra,
    )
elif USE_LM:
    sub = predict_submission_blend(text_cfg, blend_cfg, train, test, lm_train, lm_test)
else:
    sub = predict_submission(text_cfg, train, test)
sub.to_parquet("/kaggle/working/submission.parquet", index=False)
print(f"wrote submission.parquet: {len(sub)} rows in {time.time() - start:.0f}s")
sub.head()

## Runtime check

The hidden rerun has ~4,000 passages and a **9-hour** limit. On 4,000 stand-in rows
(training texts, resampled) we time the LM extraction, the encoder's 5 models and a full fit
plus prediction, then add the training-feature extraction measured above. The total must be
far below that limit.

It runs on the 3 example rows only. In the hidden rerun the real passages are the work, and
the check would repeat all of it on stand-ins, doubling the time it's meant to bound.

In [ ]:
if len(test) <= 3:
    stand_in = pd.DataFrame(
        {
            "id": [f"t{i}" for i in range(4000)],
            "text": train["text"].sample(4000, replace=True, random_state=0).to_numpy(),
        }
    )
    start = time.time()
    if USE_LM:
        lm_stand_in = lm_features(stand_in["text"].tolist())
    if USE_ENCODER:
        p_stand_in = encoder_proba(stand_in["text"].tolist())
        llm_stand_in = None
        if ens_cfg.llm is not None:
            # The LLM is the slow member: time it on 400 passages and count it 10 times, rather
            # than spend an hour of the hidden run on 4,000 stand-ins.
            llm_start = time.time()
            llm_stand_in = np.tile(llm_proba(stand_in["text"].tolist()[:400]), (10, 1))
            start -= 9 * (time.time() - llm_start)  # the 9 other tenths, as if they had run
        extra_stand_in = extra_proba(stand_in["text"].tolist()) if p_extra else None
        predict_submission_ensemble(
            text_cfg,
            blend_cfg,
            ens_cfg,
            train,
            stand_in,
            lm_train,
            lm_stand_in,
            p_stand_in,
            pool=pool,
            p_llm=llm_stand_in,
            p_extra=extra_stand_in,
        )
    elif USE_LM:
        predict_submission_blend(text_cfg, blend_cfg, train, stand_in, lm_train, lm_stand_in)
    else:
        predict_submission(text_cfg, train, stand_in)
    minutes = extract_min + (time.time() - start) / 60
    print(f"estimated hidden run: {minutes:.1f} min (limit: 540 min)")
    # The plan's bar (P7 Task 8): under 7.5 hours, against the 9-hour limit.
    assert minutes < 450, "too slow for comfort; investigate before submitting"